# 🏔️ Nepal Earthquake (2015) — Big Data Damage Assessment
**Module:** Big Data and Visualisation | **Dataset:** `eq2015.csv` (762,106 records, 21 columns)

**Analysis pipeline:** PySpark Classification → Regression → Clustering → Tableau Export

## Step 1 — Environment Setup

In [1]:
import sys, os, subprocess

# Force Java 11 — PySpark 3.x does NOT support Java 25 (which is the macOS default here)
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/microsoft-11.jdk/Contents/Home"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# Remove any broken manual Spark binary paths added by previous attempts
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]
sys.path = [p for p in sys.path if "spark-4.1.1-bin-hadoop3" not in p]

# Confirm Java 11 is active
r = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(r.stderr.strip())
print("Environment ready.")


Picked up _JAVA_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/javax.security.auth=ALL-UNNAMED
openjdk version "11.0.12" 2021-07-20
OpenJDK Runtime Environment Microsoft-25199 (build 11.0.12+7)
OpenJDK 64-Bit Server VM Microsoft-25199 (build 11.0.12+7, mixed mode)
Environment ready.


## Step 2 — Import Libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, corr, regexp_extract, count, when, isnan

from pyspark.ml.feature     import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.regression  import RandomForestRegressor
from pyspark.ml.clustering  import KMeans
from pyspark.ml.evaluation  import (MulticlassClassificationEvaluator,
                                    RegressionEvaluator, ClusteringEvaluator)
from pyspark.ml             import Pipeline

print("All libraries imported successfully.")


All libraries imported successfully.


## Step 3 — Start SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("NepalEarthquakeBigDataAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Version:", spark.version)
spark


Picked up _JAVA_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/javax.security.auth=ALL-UNNAMED
Picked up _JAVA_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/javax.security.auth=ALL-UNNAMED
26/03/09 18:01:38 WARN Utils: Your hostname, Prajeshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.90.1.20 instead (on interface en0)
26/03/09 18:01:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 18:01:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 18:01:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 3.5.0


## Step 4 — Load Dataset
Loading `eq2015.csv` using PySpark's distributed CSV reader.

In [4]:
df = spark.read.csv("eq2015.csv", header=True, inferSchema=True)
df.cache()  # Pin to memory for faster iterative queries

print(f"Total Rows    : {df.count():,}")
print(f"Total Columns : {len(df.columns)}")
df.printSchema()
df.show(5, truncate=True)


Total Rows    : 762,106
Total Columns : 21
root
 |-- building_id: long (nullable = true)
 |-- district_id: integer (nullable = true)
 |-- vdcmun_id: integer (nullable = true)
 |-- ward_id: integer (nullable = true)
 |-- count_floors_pre_eq: integer (nullable = true)
 |-- count_floors_post_eq: integer (nullable = true)
 |-- age_building: integer (nullable = true)
 |-- plinth_area_sq_ft: integer (nullable = true)
 |-- height_ft_pre_eq: integer (nullable = true)
 |-- height_ft_post_eq: integer (nullable = true)
 |-- land_surface_condition: string (nullable = true)
 |-- foundation_type: string (nullable = true)
 |-- roof_type: string (nullable = true)
 |-- ground_floor_type: string (nullable = true)
 |-- other_floor_type: string (nullable = true)
 |-- position: string (nullable = true)
 |-- plan_configuration: string (nullable = true)
 |-- condition_post_eq: string (nullable = true)
 |-- damage_grade: string (nullable = true)
 |-- technical_solution_proposed: string (nullable = true)
 |-- 

## Step 5 — Define Column Groups
Clearly separating numeric, location, and categorical features for the pipeline.

In [5]:
# Numeric features (all already integer type from schema)
num_cols = [
    "count_floors_pre_eq",
    "count_floors_post_eq",
    "age_building",
    "plinth_area_sq_ft",
    "height_ft_pre_eq",
    "height_ft_post_eq"
]

# Location identifiers (treated as categorical — indexed by StringIndexer)
loc_cols = [
    "district_id",
    "vdcmun_id",
    "ward_id"
]

# Categorical features — EXCLUDED: condition_post_eq, technical_solution_proposed
# (those are POST-earthquake outcomes → data leakage if used as model inputs)
cat_cols = [
    "land_surface_condition",
    "foundation_type",
    "roof_type",
    "ground_floor_type",
    "other_floor_type",
    "position",
    "plan_configuration",
    "superstructure"
]

print(f"Numeric   : {num_cols}")
print(f"Location  : {loc_cols}")
print(f"Categoric : {cat_cols}")


Numeric   : ['count_floors_pre_eq', 'count_floors_post_eq', 'age_building', 'plinth_area_sq_ft', 'height_ft_pre_eq', 'height_ft_post_eq']
Location  : ['district_id', 'vdcmun_id', 'ward_id']
Categoric : ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'superstructure']


## Step 6 — Handle Missing Values & Feature Engineering

In [6]:
# 6.1 Fill numeric nulls with column mean
for c in num_cols:
    mean_val = df.select(mean(col(c))).first()[0]
    df = df.na.fill({c: mean_val if mean_val is not None else 0})

# 6.2 Drop remaining rows with any nulls in cat/loc columns
df = df.dropna(subset=cat_cols + loc_cols + ["damage_grade"])

# 6.3 Regression target: extract number from "Grade X" → float (1.0–5.0)
df = df.withColumn(
    "damage_numeric",
    regexp_extract(col("damage_grade"), r"\d+", 0).cast("float")
)

print(f"Clean rows: {df.count():,}")
df.select("damage_grade", "damage_numeric").show(5)


Clean rows: 762,094
+------------+--------------+
|damage_grade|damage_numeric|
+------------+--------------+
|     Grade 3|           3.0|
|     Grade 5|           5.0|
|     Grade 2|           2.0|
|     Grade 2|           2.0|
|     Grade 1|           1.0|
+------------+--------------+
only showing top 5 rows



## Step 7 — Exploratory Data Analysis (EDA)

In [7]:
# 7.1 Summary statistics on numeric columns
print("=== Summary Statistics ===")
df.select(num_cols).describe().show()

# 7.2 Damage grade distribution
print("=== Damage Grade Distribution ===")
df.groupBy("damage_grade").count().orderBy("damage_grade").show()

# 7.3 Null check
print("=== Null Counts ===")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(truncate=False)

# 7.4 Sample correlation
print("=== Correlation: age_building vs height_ft_pre_eq ===")
df.select(corr("age_building", "height_ft_pre_eq")).show()


=== Summary Statistics ===


26/03/09 18:02:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+--------------------+------------------+------------------+------------------+-----------------+
|summary|count_floors_pre_eq|count_floors_post_eq|      age_building| plinth_area_sq_ft|  height_ft_pre_eq|height_ft_post_eq|
+-------+-------------------+--------------------+------------------+------------------+------------------+-----------------+
|  count|             762094|              762094|            762094|            762094|            762094|           762094|
|   mean| 2.0877870708862685|  1.2520502720136886|24.325030770482382|406.67366755282154|16.049424349227262|9.868785215472107|
| stddev| 0.6551040843920005|  1.0632784499065644| 65.03455481369356| 226.7804366112519| 5.493899906287996| 8.57421802889839|
|    min|                  1|                   0|                 0|                70|                 6|                0|
|    max|                  9|                   9|               999|              5000|                99|           

+------------+------+
|damage_grade| count|
+------------+------+
|     Grade 1| 78815|
|     Grade 2| 87257|
|     Grade 3|136412|
|     Grade 4|183844|
|     Grade 5|275766|
+------------+------+

=== Null Counts ===


+-----------+-----------+---------+-------+-------------------+--------------------+------------+-----------------+----------------+-----------------+----------------------+---------------+---------+-----------------+----------------+--------+------------------+-----------------+------------+---------------------------+--------------+--------------+
|building_id|district_id|vdcmun_id|ward_id|count_floors_pre_eq|count_floors_post_eq|age_building|plinth_area_sq_ft|height_ft_pre_eq|height_ft_post_eq|land_surface_condition|foundation_type|roof_type|ground_floor_type|other_floor_type|position|plan_configuration|condition_post_eq|damage_grade|technical_solution_proposed|superstructure|damage_numeric|
+-----------+-----------+---------+-------+-------------------+--------------------+------------+-----------------+----------------+-----------------+----------------------+---------------+---------+-----------------+----------------+--------+------------------+-----------------+------------+---

In [14]:
import os
from pyspark.sql.functions import count, when, col, corr

# Create a dedicated directory for EDA reports
eda_export_dir = "Tableau_Export/eda_reports"
os.makedirs(eda_export_dir, exist_ok=True)

print("🚀 Exporting additional EDA components for Tableau...")

# 7.1 Summary Statistics (Tableau: Use for descriptive stats tables)
df.select(num_cols).describe() \
    .coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{eda_export_dir}/summary_stats")

# 7.2 Damage Grade Distribution (Tableau: Perfect for a Donut or Pie chart)
df.groupBy("damage_grade").count().orderBy("damage_grade") \
    .coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{eda_export_dir}/damage_distribution")

# 7.3 Null Check Report (Tableau: Use for a Data Quality scorecard)
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]) \
    .coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{eda_export_dir}/null_counts")

# 7.4 Feature Correlation (Tableau: Use as a single KPI or Text indicator)
df.select(corr("age_building", "height_ft_pre_eq").alias("corr_age_vs_height")) \
    .coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{eda_export_dir}/correlation_analysis")

print(f"✅ Additional EDA reports ready in '{eda_export_dir}/' folder.")

🚀 Exporting additional EDA components for Tableau...


✅ Additional EDA reports ready in 'Tableau_Export/eda_reports/' folder.


## Step 8 — Feature Encoding & ML Pipeline
Chaining StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler → LabelIndexer.

In [8]:
all_cat = cat_cols + loc_cols  # encode both categorical and location cols together

# Step 1: String index all categorical columns
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in all_cat
]

# Step 2: One-hot encode (batch API — Modern PySpark 3.x style)
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in all_cat],
    outputCols=[c + "_ohe" for c in all_cat]
)

# Step 3: Assemble numeric + encoded categorical into one feature vector
assembler = VectorAssembler(
    inputCols=num_cols + [c + "_ohe" for c in all_cat],
    outputCol="features_raw",
    handleInvalid="keep"
)

# Step 4: Standardise features (critical for K-Means distance calculations)
scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                        withStd=True, withMean=False)

# Step 5: Index the damage_grade label (for classification)
label_indexer = StringIndexer(inputCol="damage_grade", outputCol="label",
                               handleInvalid="keep")

pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler, label_indexer])

print("Fitting preprocessing pipeline on 762K records...")
pipe_model     = pipeline.fit(df)
df_transformed = pipe_model.transform(df)

# 80/20 train-test split
train, test = df_transformed.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {train.count():,}")
print(f"Test  : {test.count():,}")


Fitting preprocessing pipeline on 762K records...


Train : 609,991


Test  : 152,103


## Step 9 — Classification: Random Forest
Predicting discrete damage grade category.

In [9]:
rf_classifier = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    seed=42
)

print("Training Random Forest Classifier...")
rf_clf_model      = rf_classifier.fit(train)
class_predictions = rf_clf_model.transform(test)

# Accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
).evaluate(class_predictions)

# F1
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
).evaluate(class_predictions)

print(f"\n{'='*40}")
print(f"  Classification Accuracy : {accuracy:.4f}")
print(f"  Classification F1-Score : {f1:.4f}")
print(f"{'='*40}")

# Feature importance (top-level summary for report)
print("\nFeature Importances (vector):")
print(rf_clf_model.featureImportances)


Training Random Forest Classifier...


26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_3 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_1 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_5 in memory! (computed 105.7 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_0 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_2 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_4 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN MemoryStore: Not enough space to cache rdd_278_6 in memory! (computed 69.9 MiB so far)
26/03/09 18:04:30 WARN BlockManager: Persisting block rdd_278_6 to disk instead.
26/03/09 18:04:30 WARN BlockManager: Persisting block rdd_278_5 to disk instead.
26/03/09 18:04:30 WARN BlockManager: Persisting block rdd_278_0 to 


  Classification Accuracy : 0.4267
  Classification F1-Score : 0.2779

Feature Importances (vector):
(1117,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,27,28,29,30,31,32,35,36,38,39,40,41,42,43,44,45,46,47,48,50,51,52,53,54,56,57,58,59,60,61,62,63,64,65,67,68,69,71,72,73,77,79,80,82,83,85,86,87,88,89,90,91,92,93,94,95,96,97,98,100,101,103,104,106,107,108,109,110,112,114,115,117,118,119,120,122,123,126,127,128,129,132,144,148,149,151,152,155,159,160,162,163,166,168,171,173,174,176,178,181,182,184,186,187,191,193,201,209,216,220,221,228,229,240,244,246,249,252,256,259,260,261,275,280,283,287,293,295,312,314,316,320,323,336,339,343,350,361,367,371,374,378,379,384,389,397,400,401,409,410,419,421,427,429,430,432,437,465,466,472,475,505,514,515,526,527,533,537,538,550,559,573,577,598,601,621,675,687,700,701,704,706,707,710,719,725,760,769,772,774,783,784,785,795,797,799,803,804,807,809,818,831,833,835,848,877,883,891,896,906,910,911,917,926,927,938,947,952,954,977,97

## Step 10 — Regression: Random Forest
Predicting continuous `damage_numeric` score (1.0–5.0).

In [10]:
rf_regressor = RandomForestRegressor(
    featuresCol="features",
    labelCol="damage_numeric",
    numTrees=50,
    seed=42
)

print("Training Random Forest Regressor...")
reg_model        = rf_regressor.fit(train)
reg_predictions  = reg_model.transform(test)

rmse = RegressionEvaluator(labelCol="damage_numeric", predictionCol="prediction",
                            metricName="rmse").evaluate(reg_predictions)
mae  = RegressionEvaluator(labelCol="damage_numeric", predictionCol="prediction",
                            metricName="mae").evaluate(reg_predictions)
r2   = RegressionEvaluator(labelCol="damage_numeric", predictionCol="prediction",
                            metricName="r2").evaluate(reg_predictions)

print(f"\n{'='*40}")
print(f"  RMSE : {rmse:.4f}")
print(f"  MAE  : {mae:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"{'='*40}")


Training Random Forest Regressor...


26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_4 in memory! (computed 69.9 MiB so far)
26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_1 in memory! (computed 69.9 MiB so far)
26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_5 in memory! (computed 46.6 MiB so far)
26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_6 in memory! (computed 46.6 MiB so far)
26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_7 in memory! (computed 164.1 MiB so far)
26/03/09 18:06:28 WARN MemoryStore: Not enough space to cache rdd_338_2 in memory! (computed 69.9 MiB so far)
26/03/09 18:06:28 WARN BlockManager: Persisting block rdd_338_2 to disk instead.
26/03/09 18:06:28 WARN BlockManager: Persisting block rdd_338_1 to disk instead.
26/03/09 18:06:28 WARN BlockManager: Persisting block rdd_338_4 to disk instead.
26/03/09 18:06:28 WARN BlockManager: Persisting block rdd_338_5 to disk instead.
26/03/09 18:06:


  RMSE : 0.7327
  MAE  : 0.5029
  R²   : 0.7015


## Step 11 — Clustering: K-Means (k=5)
Unsupervised segmentation on the full dataset (not just train) to discover natural vulnerability profiles.

In [11]:
# NOTE: Clustering runs on FULL transformed dataset — not just train
# This is intentional: clustering is exploratory, not predictive
kmeans = KMeans(featuresCol="features", k=5, seed=42)

print("Training K-Means (k=5) on full dataset...")
kmeans_model   = kmeans.fit(df_transformed)
cluster_results = kmeans_model.transform(df_transformed) \
                              .withColumnRenamed("prediction", "cluster")

# Silhouette score
sil = ClusteringEvaluator(featuresCol="features", predictionCol="cluster"
                           ).evaluate(cluster_results)

print(f"\n{'='*40}")
print(f"  Silhouette Score : {sil:.4f}")
print(f"{'='*40}")

print("\nBuildings per cluster:")
cluster_results.groupBy("cluster").count().orderBy("cluster").show()

print("\nAverage damage score per cluster:")
cluster_results.groupBy("cluster").agg(
    mean(col("damage_numeric")).alias("avg_damage"),
    mean(col("age_building")).alias("avg_age"),
    mean(col("plinth_area_sq_ft")).alias("avg_plinth_sqft")
).orderBy("cluster").show()


Training K-Means (k=5) on full dataset...


26/03/09 18:09:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS



  Silhouette Score : -0.1594

Buildings per cluster:


+-------+------+
|cluster| count|
+-------+------+
|      0|757848|
|      1|   959|
|      2|  1144|
|      3|  1181|
|      4|   962|
+-------+------+


Average damage score per cluster:


+-------+------------------+------------------+------------------+
|cluster|        avg_damage|           avg_age|   avg_plinth_sqft|
+-------+------------------+------------------+------------------+
|      0| 3.648498115717136|24.361162132775966|406.91015348724284|
|      1|2.3034410844629822|14.093847758081335| 370.0500521376434|
|      2| 3.804195804195804|21.926573426573427|290.85402097902096|
|      3|2.6613039796782387| 23.52497883149873| 505.2921253175275|
|      4| 2.141372141372141| 9.895010395010395|273.54573804573806|
+-------+------------------+------------------+------------------+



## Step 12 — Export Results for Tableau
Three separate CSV exports: classification predictions, clustering assignments, and regression predictions.

In [12]:
import os
OUTPUT_DIR = "Tableau_Export"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 12.1 Classification results
print("Exporting classification results...")
class_predictions.select("building_id", "district_id", "damage_grade", "label", "prediction") \
    .coalesce(1).write.mode("overwrite").csv(f"{OUTPUT_DIR}/classification", header=True)

# 12.2 Clustering results
print("Exporting clustering results...")
cluster_results.select("building_id", "district_id", "vdcmun_id",
                        "damage_grade", "damage_numeric", "cluster",
                        "foundation_type", "roof_type", "age_building") \
    .coalesce(1).write.mode("overwrite").csv(f"{OUTPUT_DIR}/clustering", header=True)

# 12.3 Regression results
print("Exporting regression results...")
reg_predictions.select("building_id", "district_id", "damage_grade",
                        "damage_numeric", "prediction") \
    .withColumnRenamed("prediction", "predicted_severity") \
    .coalesce(1).write.mode("overwrite").csv(f"{OUTPUT_DIR}/regression", header=True)

print(f"\n✅ All 3 exports complete! Check the '{OUTPUT_DIR}/' folder.")
print("Import those 3 CSVs into Tableau to build your dashboard.")


Exporting classification results...


Exporting clustering results...


Exporting regression results...



✅ All 3 exports complete! Check the 'Tableau_Export/' folder.
Import those 3 CSVs into Tableau to build your dashboard.


In [16]:
import os
from pyspark.sql.functions import col, avg, count, floor, when

# Ensure you have your 'df' (cleaned main data), 'class_predictions', 
# 'reg_predictions', and 'cluster_results' dataframes available in your Spark session.

EXPORT_DIR = "Tableau_Exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Classification (Confusion Matrix Prep)
class_predictions.select("building_id", "damage_grade", "prediction") \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset1_classification")

# 2. Severity Trends (Regression Stats)
reg_predictions.select("building_id", "damage_numeric", "prediction") \
    .withColumnRenamed("prediction", "predicted_severity") \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset2_regression")

# 3. Vulnerability Profiling (Cluster Stats)
cluster_results.select("building_id", "cluster", "damage_numeric", "damage_grade") \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset3_clustering")

# 4. Regional Heatmap (Aggregated by District)
df.groupBy("district_id") \
    .agg(avg("damage_numeric").alias("avg_damage_severity")) \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset4_geospatial")

# 5. Structural Breakdown (Foundation & Roofs)
df.groupBy("foundation_type", "roof_type", "damage_grade") \
    .count().write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset5_structural")

# 6. Building Age Impact (Binned Age Groups)
df.withColumn("age_bin", floor(col("age_building") / 10) * 10) \
    .groupBy("age_bin").agg(avg("damage_numeric").alias("avg_damage")) \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset6_temporal")

# 7. Material Resistance (Superstructure Summary)
df.groupBy("superstructure") \
    .agg(count("building_id").alias("building_count"), avg("damage_numeric").alias("avg_damage_numeric")) \
    .write.mode("overwrite").option("header", "true").csv(f"{EXPORT_DIR}/subset7_superstructure")

print(f"✅ All 7 datasets exported successfully to the '{EXPORT_DIR}/' folder.")

✅ All 7 datasets exported successfully to the 'Tableau_Exports/' folder.


In [15]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Forensic Engineering: Calculate Height Loss & Compression Ratio
# Metrics for: "The Pancake Effect" and "Height Differential" analysis
df_eda = df.withColumn("height_loss", F.col("height_ft_pre_eq") - F.col("height_ft_post_eq")) \
           .withColumn("compression_ratio", 
                       F.when(F.col("height_ft_pre_eq") > 0, 
                              F.col("height_ft_post_eq") / F.col("height_ft_pre_eq")).otherwise(0))

# 2. Structural Profiling: Material Lethality Index
# Calculates the % of buildings that collapsed (Grade 5) per category
lethality_window = Window.partitionBy("superstructure")
df_eda = df_eda.withColumn("is_grade_5", F.when(F.col("damage_grade") == "Grade 5", 1).otherwise(0)) \
               .withColumn("material_lethality_index", F.avg("is_grade_5").over(lethality_window))

# 3. Code Adherence Analysis: Building Age Binning
# Creates 10-year "epochs" to find the "Year of Safety" in Tableau
df_eda = df_eda.withColumn("age_bin", (F.floor(F.col("age_building") / 10) * 10).cast("string"))

# 4. Geotechnical Risk: Flagging Steep Slopes vs Height
# Useful for the Land Surface Condition scatter plots
df_eda = df_eda.withColumn("is_steep_slope", F.when(F.col("land_surface_condition") == "Steep slope", 1).otherwise(0))

# 5. ML Cluster Profiling: Join Cluster IDs (if available)
# Assuming you have a 'cluster_results' dataframe from your K-Means step
if 'cluster' in globals():
    df_eda = df_eda.join(cluster_results.select("building_id", "cluster"), on="building_id", how="left")

# 6. Technical Solution Urgency
# Assigns a weight to solutions to visualize recovery cost in Tableau
solution_weights = {
    "Reconstruction": 5,
    "Major repair": 3,
    "Minor repair": 1,
    "No need": 0
}
mapping_expr = F.create_map([F.lit(x) for x in sum(solution_weights.items(), ())])
df_eda = df_eda.withColumn("recovery_urgency_score", mapping_expr[F.col("technical_solution_proposed")])

# 7. Final Selection & Export
# Selecting a balanced mix of raw features and new engineered metrics
final_tableau_df = df_eda.select(
    "building_id", "district_id", "damage_grade", "damage_numeric",
    "foundation_type", "superstructure", "plan_configuration", "roof_type",
    "age_bin", "plinth_area_sq_ft", "height_ft_pre_eq", "height_ft_post_eq",
    "height_loss", "compression_ratio", "material_lethality_index", 
    "is_steep_slope", "recovery_urgency_score"
)

# Export as a single CSV for easy loading into Tableau
final_tableau_df.coalesce(1).write.mode("overwrite").csv("Tableau_Advanced_EDA_Export", header=True)

print("✅ Advanced EDA Export ready for Tableau.")

✅ Advanced EDA Export ready for Tableau.
